# Phase 2 (awake reference): which LFP-tracker ↔ spiking relationships hold?

Fixed catalog of relationships (`src/relationships.py`), tested on awake, stimulation-free seconds with the
Phase 1 units and ensembles. Mice were split at random **before any relationship was computed**
(`data/processed/mouse_split.json`): discovery 688277, 692644, 703063, 720762; confirmation 703064, 703065, 709401.
Only relationships that passed Benjamini–Hochberg in discovery were tested in confirmation, one-sided in the
discovery direction, again with BH. 688277 has too little awake data for the tracker, so discovery has 3 mice
with tracker motifs (17 signals); confirmation has 3 (25 signals).

**Controls in every test.** Pupil width and running speed are partialled out (arousal covariates; confounded with
each other and, later, with isoflurane dose). Nulls circularly shift the LFP side against the spikes; all signals
recorded at the same time share the same shifts, so they are not counted as independent. Effects are pooled as the
mean over signals; "excess" = effect minus the null mean.

**What each relationship means**
- R1: does the motif's slow baseline go with overall firing (population, + and − ensembles)?
- R2: per neuron, does its coupling to the motif's fast expression change with the baseline (the expression ×
  baseline interaction)? Share of neurons with |t| > 2, against a null where only the interaction's baseline is
  shifted. This is the "neurons join or leave as the baseline moves" hypothesis.
- R4: does the + minus − ensemble contrast rise with expression (intensity), and does that slope change with the
  baseline (gain)?
- R5: in windows where the motif's poles sit further apart, are neurons more or less coupled / more opposed?
- R6: does pole asymmetry or midpoint go with the + vs − balance?
- R7: does spread go with synchrony, dimensionality, Fano factor?
- R3 / R8: does baseline movement (speed, rotation of the 10-band baseline vector) go with ensemble turnover?
- R9: does presence go with how many neurons are coupled?
- R10: are 2-min windows that look alike on the LFP side (baseline vector, pole positions, spread) also alike in
  spiking (rates, synchrony, dimensionality, Fano, which neurons are coupled)? Mantel r.
- R11: timing of the peak cross-correlation between expression and the ensemble contrast.
- R12: do the tracker's outputs explain population firing and the ensemble contrast beyond pupil + running?

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import awake_grounding as ag
P = REPO / "data/processed/relationships_awake"
OUT = REPO / "data" / "out" / "2026-11-15_relationships_awake"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
D = pd.read_pickle(P / "discovery_pooled.pkl"); C = pd.read_pickle(P / "confirmation_pooled.pkl")
M = pd.read_pickle(P / "confirmation_tests.pkl")
D.drop(columns="per mouse excess").to_csv(OUT / "discovery_all_relationships.csv", index=False)
M.drop(columns="per mouse excess").to_csv(OUT / "confirmation_tests.csv", index=False)
D.drop(columns="per mouse excess").round(3)

,level,relationship,signals,mean effect,null mean,excess over null,p,mice same sign,BH significant
0,second,R1 baseline~minus rate,16,0.071,-0.014,0.085,0.000,2/3,True
1,second,R1 baseline~plus rate,15,-0.043,-0.011,-0.032,0.543,1/3,False
2,second,R1 baseline~pop rate,17,-0.073,0.016,-0.089,0.000,2/3,True
3,second,R12 R2 gain over pupil+running (contrast),15,0.134,-0.012,0.146,0.000,3/3,True
4,second,R12 R2 gain over pupil+running (pop),17,0.163,0.002,0.160,0.000,2/3,True
5,second,R2 neurons whose motif coupling changes with b...,17,0.589,0.407,0.182,0.000,3/3,True
6,second,R4 contrast slope on expression,15,0.503,0.002,0.502,0.000,3/3,True
7,second,R4 gain change of that slope with baseline,15,0.086,-0.007,0.093,0.000,3/3,True
8,window,R10 umbrella (Mantel r),17,0.432,-0.051,0.483,0.000,3/3,True
9,window,R1w baseline ~ population rate,17,-0.047,0.015,-0.062,0.520,2/3,False


## Discovery → confirmation
Each row: one relationship. Dots: per-mouse excess over the null (filled = confirmation mice, open = discovery
mice); bars: pooled excess. Rows that failed BH in discovery are shown greyed for completeness.

In [2]:
order = D.sort_values(["BH significant", "excess over null"], ascending=[False, False]).relationship.tolist()
fig, ax = plt.subplots(figsize=(10, 0.42 * len(order) + 1.2))
for i, rel in enumerate(order):
    d = D[D.relationship == rel].iloc[0]
    sig = bool(d["BH significant"])
    ax.scatter(list(d["per mouse excess"].values()), [i - 0.12] * len(d["per mouse excess"]), s=26, facecolor="white",
               edgecolor="#2a78d6" if sig else "0.7", lw=1.1, zorder=3)
    ax.plot([0, d["excess over null"]], [i - 0.12] * 2, color="#2a78d6" if sig else "0.8", lw=3, alpha=0.5)
    c = M[M.relationship == rel]
    if len(c):
        c = c.iloc[0]
        col = "#1baf7a" if c["confirmed (BH)"] else "#eb6834"
        ax.scatter(list(c["per mouse excess"].values()), [i + 0.12] * len(c["per mouse excess"]), s=26, color=col, zorder=3)
        ax.plot([0, c["excess over null"]], [i + 0.12] * 2, color=col, lw=3, alpha=0.6)
        ax.text(1.02, i, "confirmed" if c["confirmed (BH)"] else "not confirmed", transform=ax.get_yaxis_transform(),
                fontsize=8, va="center", color=col)
ax.axvline(0, color="0.5", lw=0.8)
ax.set_yticks(range(len(order)), order, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("excess over null (r, R², share or slope, depending on the relationship)")
ax.set_title("Upper (blue/grey, open): discovery mice. Lower (green = confirmed, orange = not): confirmation mice", fontsize=9, loc="left")
save(fig, "1_discovery_confirmation"); plt.show()
M.drop(columns="per mouse excess").round(4)

,level,relationship,discovery excess,discovery mice,signals,excess over null,p,mice same sign,p one-sided,confirmed (BH)
0,second,R1 baseline~minus rate,0.0852,2/3,25,0.0689,0.0014,2/3,0.0007,True
1,second,R1 baseline~pop rate,-0.0892,2/3,25,-0.0267,0.0792,2/3,0.0396,True
2,second,R12 R2 gain over pupil+running (contrast),0.1464,3/3,25,0.1565,0.0002,3/3,0.0001,True
3,second,R12 R2 gain over pupil+running (pop),0.1604,2/3,25,0.0915,0.0002,3/3,0.0001,True
4,second,R2 neurons whose motif coupling changes with b...,0.1823,3/3,25,0.1426,0.0002,3/3,0.0001,True
5,second,R4 contrast slope on expression,0.5018,3/3,25,0.4553,0.0002,3/3,0.0001,True
6,second,R4 gain change of that slope with baseline,0.0930,3/3,25,0.0073,0.3751,2/3,0.1876,False
7,window,R10 umbrella (Mantel r),0.4826,3/3,25,0.3200,0.0002,3/3,0.0001,True
8,window,R5 separation ~ coupling strength,0.6176,3/3,25,0.5082,0.0002,3/3,0.0001,True


## Timing (R11)
Lag of the peak cross-correlation between motif expression and the + minus − ensemble contrast, per signal
(1-s resolution; negative = spiking leads the LFP envelope).

In [3]:
lags = {g: [o["timing"].get("R11 lag of peak (s; + = spiking follows LFP)") for o in ag.load(P / f"{g}.pkl").values()] for g in ("discovery", "confirmation")}
lags = {g: [l for l in v if l is not None] for g, v in lags.items()}
pd.DataFrame({g: pd.Series(v).value_counts().sort_index() for g, v in lags.items()}).fillna(0).astype(int)

,discovery,confirmation
-2.0,0,2
-1.0,6,12
0.0,7,10
1.0,2,1


## Conclusions
Confirmed in held-out mice, same sign in all 3 confirmation mice (awake, pupil and running partialled out):
- **R2 — neurons' coupling to the motif changes with the baseline (the join/leave hypothesis).** 59 % of neurons
  show an expression × baseline interaction vs 41 % expected (discovery); excess 0.14 in confirmation, 3/3 mice.
- **R4 — intensity.** The + minus − ensemble contrast rises with motif expression (slope ≈ 0.5 SD per SD).
- **R5 — pole separation goes with coupling strength.** In windows where the poles sit further apart, coupled
  neurons follow the motif more strongly (r ≈ 0.5). Caveat: larger LFP variance alone raises correlations; a
  control with a non-motif direction is needed before reading this as biology.
- **R10 — umbrella.** Windows that look alike on the LFP side are alike in spiking (Mantel r 0.32 over the null).
- **R12 — beyond arousal.** The tracker's outputs explain the ensemble contrast (+0.16 R²) and population firing
  (+0.09 R²) beyond pupil and running.

Not a consistent theme: **R1** (baseline vs firing level) passes BH but its sign differs between mice.
Not confirmed: **R4 gain** — the ensemble-level slope does not change with the baseline. Together with R2 this fits
membership turnover rather than a uniform gain change: individual neurons' coupling changes in both directions,
which cancels at the ensemble level.
Nothing in R3, R6–R9 passed discovery; baseline movement within 15 min of awake data is small, so these are
better tested under isoflurane. **Timing (R11):** peaks at −1 to 0 s, i.e. spiking leads or coincides with the
LFP envelope at 1-s resolution.